In [ ]:
# -*- coding: utf-8 -*-

# Symmetry - Time - Robotics - Quantum Mechanics
*The same Lie group SO(3) describes electron spin and robot arm rotation*

In [ ]:
import sys, io
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8', errors='replace')
sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding='utf-8', errors='replace')

import sympy as sp
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sympy import *
sp.init_printing(use_latex="mathjax")

try:
    from IPython.display import display as _ipy_display
    def show(expr, label=None):
        if label: print(f"  {label}:")
        _ipy_display(expr)
except ImportError:
    def show(expr, label=None):
        if label: print(f"  {label}:")
        print("  " + sp.pretty(expr, use_unicode=True))

def hdr(s): print(f"\n{'='*60}\n  {s}\n{'='*60}")

def chk(val, ref, label, tol=1e-8, absolute=False):
    v, r = float(np.real(val)), float(np.real(ref))
    err = abs(v-r) if (absolute or abs(r)<1e-30) else abs(v-r)/(abs(r)+1e-30)
    s = "PASS" if err < tol else "FAIL"
    print(f"  [{s}] {label}: got {v:.6g}, ref {r:.6g}, err {err:.2e}")
    return s == "PASS"

## §1 — Symmetry groups: discrete and continuous

In [ ]:
hdr("§1 — Symmetry groups: discrete and continuous")

# --- Z_n: rotations of regular n-gon ---
print("Z_3 multiplication table (elements: 0,1,2 representing e,r,r^2):")
n = 3
Z3_table = np.array([[( i + j) % n for j in range(n)] for i in range(n)])
print(Z3_table)

# Group axioms for Z_4
print("\nVerifying Z_4 group axioms:")
n4 = 4
elements = list(range(n4))
# Closure: all products are in Z_4
Z4_products = [(i+j) % n4 for i in elements for j in elements]
Z4_closure_count = len(Z4_products)  # 16 products
all_in_Z4 = all(p in elements for p in Z4_products)
print(f"  Closure: {all_in_Z4}, product count={Z4_closure_count}")
# Identity: 0
identity_ok = all((i+0)%n4 == i for i in elements)
print(f"  Identity (e=0): {identity_ok}")
# Inverses: each element has (-i)%4
inverses_ok = all((i + (-i)%n4) % n4 == 0 for i in elements)
print(f"  Inverses: {inverses_ok}")

# --- Dihedral group D_3 ---
from sympy.combinatorics import DihedralGroup
D3 = DihedralGroup(3)
D3_order = D3.order()
print(f"\nD_3 order = {D3_order}")

# Non-abelian demonstration: r·s ≠ s·r
# Using permutation elements: rotation r = (0 1 2), reflection s = (1 2)
from sympy.combinatorics import Permutation
r_perm = Permutation(0, 1, 2)   # rotation
s_perm = Permutation(1, 2)       # reflection (fixes 0, swaps 1,2)
rs = r_perm * s_perm
sr = s_perm * r_perm
print(f"  r·s = {rs}, s·r = {sr}, r·s == s·r: {rs == sr}")

# --- SO(2) ---
theta = symbols('theta', real=True)
R2 = Matrix([[cos(theta), -sin(theta)], [sin(theta), cos(theta)]])
det_R2 = det(R2)
ortho_check = simplify(R2.T - R2**(-1))
L_gen = diff(R2, theta).subs(theta, 0)
print("\nSO(2):")
show(R2, "R(theta)")
show(det_R2, "det(R)")
show(L_gen, "Generator L = dR/dtheta|_{theta=0}")

det_R2_at_pi3 = float(det_R2.subs(theta, pi/3))

# --- SO(3) Rodrigues ---
print("\nSO(3) Rodrigues for 90° around z:")
nx, ny, nz = 0, 0, 1
K = np.array([[0, -nz, ny], [nz, 0, -nx], [-ny, nx, 0]], dtype=float)
angle = np.pi/2
R_rod = np.eye(3) + np.sin(angle)*K + (1-np.cos(angle))*(K@K)
print(f"  R_z(90°) =\n{np.round(R_rod, 6)}")

chk(Z4_closure_count, 16, "Z4_closure count==16", tol=1e-10, absolute=True)
chk(D3_order, 6, "D3_order==6", tol=1e-10, absolute=True)
chk(det_R2_at_pi3, 1.0, "det_R2 at theta=pi/3", tol=1e-10, absolute=True)
chk(R_rod[0,1], -1.0, "Rodrigues_z90 R[0,1]==-1", tol=1e-10, absolute=True)
chk(R_rod[1,0],  1.0, "Rodrigues_z90 R[1,0]==1",  tol=1e-10, absolute=True)

## §2 — Time-reversal symmetry: T, P, C, CPT

In [ ]:
hdr("§2 — Time-reversal symmetry: T, P, C, CPT")

print("Parity P: position odd, momentum odd, angular momentum even (pseudovector)")
print("Time reversal T: anti-unitary; T²=-1 for spin-1/2 (Kramers degeneracy)")
print("CPT theorem: ALL local Lorentz-invariant QFTs are CPT invariant")
print("CP violation → T violation (to preserve CPT); observed in B mesons (BaBar 2012)")

# Schrödinger equation under T
t_sym, hbar, m_sym = symbols('t hbar m', positive=True)
psi = Function('psi')
H_op = symbols('H')
print("\nSchrödinger: iℏ ∂ψ/∂t = Ĥψ; under T (t→-t, ψ→ψ*) → same eq for ψ* at -t ✓")

# Arrow of time — entropy
k_B = 1.380649e-23
Omega_1, Omega_2 = 1, 100
S1 = k_B * np.log(Omega_1)
S2 = k_B * np.log(Omega_2)
dS = S2 - S1
print(f"\nEntropy: S1=k_B ln(1)={S1:.3e}, S2=k_B ln(100)={S2:.3e}, dS={dS:.3e} > 0")

# T-symmetry: harmonic oscillator forward then backward
from scipy.integrate import solve_ivp

def harmonic_osc(t, y, omega=1.0):
    q, p = y
    return [p, -omega**2 * q]

omega = 1.0
y0 = [1.0, 0.0]
t_end = 10.0

sol_fwd = solve_ivp(harmonic_osc, [0, t_end], y0, dense_output=True, rtol=1e-10, atol=1e-12)
q_mid = sol_fwd.y[0, -1]
p_mid = sol_fwd.y[1, -1]

# Reverse: flip momentum, integrate forward (equivalent to backward in time)
y_rev = [q_mid, -p_mid]
sol_bwd = solve_ivp(harmonic_osc, [0, t_end], y_rev, dense_output=True, rtol=1e-10, atol=1e-12)
q_final = sol_bwd.y[0, -1]
p_final = -sol_bwd.y[1, -1]  # flip back

print(f"\nT-symmetry harmonic oscillator: q(0)=1, p(0)=0 → t=10 → reverse → back")
print(f"  q_final={q_final:.8f}, p_final={p_final:.8f}")

# CPT mass equality
m_electron = 9.10938e-31
m_positron = 9.10938e-31  # CPT guarantees equality
cpt_ratio = m_electron / m_positron

chk(q_final, 1.0, "T_reversal_q_final near 1.0", tol=1e-4, absolute=True)
chk(p_final, 0.0, "T_reversal_p_final near 0.0", tol=1e-4, absolute=True)
chk(cpt_ratio, 1.0, "CPT_mass_equality m_e==m_positron", tol=1e-10, absolute=True)
chk(float(dS > 0), 1.0, "entropy_arrow dS>0", tol=1e-10, absolute=True)

## §3 — Noether theorem: symmetry → conservation (formal)

In [ ]:
hdr("§3 — Noether theorem: symmetry → conservation")

q_sym, qdot, qddot = symbols('q qdot qddot', real=True)
m_n, k_n = symbols('m k', positive=True)

L_harm = Rational(1, 2)*m_n*qdot**2 - Rational(1, 2)*k_n*q_sym**2

p_mom = diff(L_harm, qdot)
H_en = p_mom*qdot - L_harm
H_en_simplified = expand(H_en)

show(L_harm, "L (harmonic oscillator)")
show(p_mom, "p = ∂L/∂q̇ (momentum)")
show(H_en_simplified, "H = p·q̇ - L (energy)")

# Verify dH/dt = 0 using EL: p_dot = -k*q, qddot = p_dot/m = -k*q/m
p_dot_el = -k_n * q_sym
qddot_el = p_dot_el / m_n  # = -k*q/m

dH_dt = diff(H_en_simplified, q_sym)*qdot + diff(H_en_simplified, qdot)*qddot
dH_dt_el = dH_dt.subs(qddot, qddot_el)
dH_dt_el_simplified = simplify(dH_dt_el)
print(f"\n  dH/dt (on-shell) = {dH_dt_el_simplified}  (should be 0)")

print("\nGauge symmetry Noether: U(1) local gauge invariance → photon + EM current conservation")
print("  ψ → e^{iα(x)}ψ forces covariant derivative D_μ = ∂_μ - ieA_μ → photon exists!")

# Numerical checks
p_num = float(p_mom.subs([(m_n, 1), (qdot, 3)]))
H_num = float(H_en_simplified.subs([(q_sym, 1), (qdot, 2), (m_n, 1), (k_n, 1)]))
dH_num = float(dH_dt_el_simplified.subs([(q_sym, 1), (qdot, 1), (m_n, 1), (k_n, 1)]))

chk(p_num, 3.0, "p_momentum at m=1,qdot=3 == 3", tol=1e-10, absolute=True)
chk(H_num, 2.5, "H_energy at q=1,qdot=2,m=1,k=1 == 2.5", tol=1e-10, absolute=True)
chk(dH_num, 0.0, "dH_dt_zero on-shell == 0", tol=1e-10, absolute=True)

## §4 — Gauge symmetry: U(1) → EM, SU(2) → weak force

In [ ]:
hdr("§4 — Gauge symmetry")

Ex, Ey, Ez, Bx, By, Bz = symbols('Ex Ey Ez Bx By Bz', real=True)
F_em = Matrix([
    [0,   Ex,  Ey,  Ez],
    [-Ex,  0,  -Bz,  By],
    [-Ey,  Bz,  0,  -Bx],
    [-Ez, -By,  Bx,   0]
])
show(F_em, "F_μν (Maxwell tensor)")

antisym_check = simplify(F_em.T + F_em)
F01_plus_F10 = float(F_em[0,1] + F_em[1,0])  # should be 0 (symbolic: Ex - Ex = 0)
# evaluate numerically
F_num = F_em.subs([(Ex,1),(Ey,2),(Ez,3),(Bx,4),(By,5),(Bz,6)])
F01_F10_num = float(F_num[0,1] + F_num[1,0])

print("\nGauge groups and force carriers:")
print("  U(1):   1 generator  → photon         → QED")
print("  SU(2):  3 generators → W+, W-, Z      → Weak force")
print("  SU(3):  8 generators → 8 gluons       → QCD")
print("  SM gauge group: SU(3)×SU(2)×U(1), rank = 8+3+1 = 12 generators")

SU2_generators = 3
SU3_generators = 8
SM_rank = 1 + SU2_generators + SU3_generators

chk(F01_F10_num, 0.0, "F_antisymmetric F[0,1]+F[1,0]==0", tol=1e-10, absolute=True)
chk(SU2_generators, 3, "SU2_generators_count==3", tol=1e-10, absolute=True)
chk(SU3_generators, 8, "SU3_generators_count==8", tol=1e-10, absolute=True)
chk(SM_rank, 12, "SM_gauge_group_rank 12 generators", tol=1e-10, absolute=True)

## §5 — Lie groups in robotics: SO(3) and SE(3)

In [ ]:
hdr("§5 — Lie groups in robotics: SO(3) and SE(3)")

# SO(3): Rodrigues
print("SO(3): 3x3 real matrices, det=+1, R^T R=I, dim=3")
print("Lie algebra so(3): skew-symmetric matrices (skew-sym of ω vector)")

# SE(3) element
R_z90 = Matrix([[0, -1, 0], [1, 0, 0], [0, 0, 1]])
p_vec = Matrix([1, 2, 3])

T_se3 = Matrix.zeros(4, 4)
T_se3[:3, :3] = R_z90
T_se3[:3, 3] = p_vec
T_se3[3, 3] = 1
show(T_se3, "T ∈ SE(3)")

det_R_z90 = float(det(R_z90))
det_T_3x3 = float(det(T_se3[:3, :3]))

# T inverse
R_inv = R_z90.T
p_inv = -R_inv * p_vec
T_inv = Matrix.zeros(4, 4)
T_inv[:3, :3] = R_inv
T_inv[:3, 3] = p_inv
T_inv[3, 3] = 1

T_prod = T_se3 * T_inv
T_prod_00 = float(T_prod[0, 0])

chk(det_R_z90, 1.0, "det_R_z90==1", tol=1e-10, absolute=True)
chk(det_T_3x3, 1.0, "T_se3_det3x3==1", tol=1e-10, absolute=True)
chk(float(T_se3[3, 0]), 0.0, "T_se3_bottom T[3,0]==0", tol=1e-10, absolute=True)
chk(float(T_se3[3, 3]), 1.0, "T_se3_bottom T[3,3]==1", tol=1e-10, absolute=True)
chk(T_prod_00, 1.0, "SE3_inverse T@T_inv=I [0,0]==1", tol=1e-10, absolute=True)

## §6 — Denavit-Hartenberg convention

In [ ]:
hdr("§6 — Denavit-Hartenberg convention")

theta_d, d_d, a_d, alpha_d = symbols('theta d a alpha', real=True)

ct, st = cos(theta_d), sin(theta_d)
ca, sa = cos(alpha_d), sin(alpha_d)

T_DH = Matrix([
    [ct, -st*ca,  st*sa, a_d*ct],
    [st,  ct*ca, -ct*sa, a_d*st],
    [0,   sa,    ca,    d_d   ],
    [0,   0,     0,     1     ]
])
show(T_DH, "DH transformation matrix")

# Verify identity at θ=d=a=α=0
T_DH_identity = T_DH.subs([(theta_d, 0), (d_d, 0), (a_d, 0), (alpha_d, 0)])
T_DH_id_00 = float(T_DH_identity[0, 0])

# 2-DOF planar robot arm
theta1_v = np.radians(30)
theta2_v = np.radians(45)
L1_v, L2_v = 1.0, 0.8

c1, s1 = np.cos(theta1_v), np.sin(theta1_v)
c2, s2 = np.cos(theta2_v), np.sin(theta2_v)
c12 = np.cos(theta1_v + theta2_v)
s12 = np.sin(theta1_v + theta2_v)

x_ee_v = L1_v * c1 + L2_v * c12
y_ee_v = L1_v * s1 + L2_v * s12
print(f"\n2-DOF planar robot: θ1=30°, θ2=45°, L1=1, L2=0.8")
print(f"  x_ee = {x_ee_v:.6f}")
print(f"  y_ee = {y_ee_v:.6f}")

chk(T_DH_id_00, 1.0, "T_DH_identity [0,0]==1", tol=1e-10, absolute=True)
chk(x_ee_v, 1.073, "x_ee vs 1.073", tol=0.001, absolute=True)
chk(y_ee_v, 1.273, "y_ee vs 1.273", tol=0.001, absolute=True)

## §7 — Robot Jacobian: velocity kinematics and singularities

In [ ]:
hdr("§7 — Robot Jacobian")

theta1, theta2, L1s, L2s = symbols('theta1 theta2 L1 L2', real=True)
x_ee_sym = L1s*cos(theta1) + L2s*cos(theta1+theta2)
y_ee_sym = L1s*sin(theta1) + L2s*sin(theta1+theta2)

J_sym = Matrix([
    [diff(x_ee_sym, theta1), diff(x_ee_sym, theta2)],
    [diff(y_ee_sym, theta1), diff(y_ee_sym, theta2)]
])
J_simplified = simplify(J_sym)
show(J_simplified, "Jacobian J")

det_J_sym = simplify(det(J_sym))
show(det_J_sym, "det(J)")

# Numerical at θ1=30°, θ2=45°, L1=1, L2=0.8
J_num = np.array([
    [-L1_v*s1 - L2_v*s12, -L2_v*s12],
    [ L1_v*c1 + L2_v*c12,  L2_v*c12]
])
det_J_num = np.linalg.det(J_num)
svd_vals = np.linalg.svd(J_num, compute_uv=False)
cond_num = svd_vals[0] / svd_vals[-1]
print(f"\n  det(J) at θ1=30°,θ2=45° = {det_J_num:.6f}")
print(f"  Condition number κ = {cond_num:.4f}")

# det(J) symbolic at L1=1, L2=0.8, θ2=π/4
det_J_val = float(det_J_sym.subs([(L1s,1),(L2s,0.8),(theta1,theta1_v),(theta2,theta2_v)]))
det_J_ref = 0.8 * np.sin(np.pi/4)

# Singular at θ2=0
J_sing = np.array([
    [-L1_v*np.sin(0) - L2_v*np.sin(0),   -L2_v*np.sin(0)],
    [ L1_v*np.cos(0) + L2_v*np.cos(0),    L2_v*np.cos(0)]
])
det_J_sing = abs(np.linalg.det(J_sing))

chk(det_J_val, det_J_ref, "det_J_symbolic vs L1*L2*sin(t2)", tol=1e-6, absolute=False)
chk(det_J_sing, 0.0, "det_J_singular_at_t2_zero < 1e-10", tol=1e-10, absolute=True)
chk(cond_num, 50.0, "condition_number_normal finite (<100)", tol=50.0, absolute=False)

## §8 — SU(2) ↔ SO(3): the quantum-robotics bridge

In [ ]:
hdr("§8 — SU(2) ↔ SO(3): the quantum-robotics bridge")

print("SU(2) → SO(3): 2:1 covering map; U and -U give same rotation")
print("Spin-1/2 needs 4π rotation (not 2π) to return to original state")
print("SU(2) ≅ S³ (3-sphere); SO(3) ≅ RP³ (projective 3-space)")
print("Quaternions = SU(2): q = a+bi+cj+dk ↔ 2×2 complex unitary matrix")

# SU(2) matrix symbolic
a_c, b_c = symbols('a b', complex=True)
U_su2 = Matrix([[a_c, -conjugate(b_c)], [b_c, conjugate(a_c)]])
show(U_su2, "U ∈ SU(2)")

# Numerical check: a=cos(0.3), b=sin(0.3)
a_num = np.cos(0.3) + 0j
b_num = np.sin(0.3) + 0j
U_num = np.array([[a_num, -np.conj(b_num)], [b_num, np.conj(a_num)]])
UU_dag = U_num @ U_num.conj().T
det_U = np.linalg.det(U_num)
UU_00 = float(np.real(UU_dag[0, 0]))

# Spin-1/2: 2π rotation gives -I, 4π gives +I
sigma_z = np.array([[1, 0], [0, -1]])
U_2pi = np.array([[np.exp(1j*np.pi), 0], [0, np.exp(-1j*np.pi)]])   # exp(i*pi*sigma_z)
U_4pi = np.array([[np.exp(2j*np.pi), 0], [0, np.exp(-2j*np.pi)]])  # 4π

U_2pi_00 = float(np.real(U_2pi[0, 0]))   # should be -1
U_4pi_00 = float(np.real(U_4pi[0, 0]))   # should be +1

print(f"\n  U(2π)|₀₀ = {U_2pi_00:.6f}  (should be -1, spin gets minus sign)")
print(f"  U(4π)|₀₀ = {U_4pi_00:.6f}  (should be +1, returns to identity)")

print("\nPauli matrices = generators of SU(2) ≡ generators of SO(3)")
print("  [J_i, J_j] = i ε_ijk J_k — SAME algebra in QM spin and robot rotations!")

chk(UU_00, 1.0, "SU2_unitary (U@U†)[0,0]==1", tol=1e-10, absolute=True)
chk(float(np.real(det_U)), 1.0, "SU2_det==1", tol=1e-10, absolute=True)
chk(U_4pi_00, 1.0, "spin_half_4pi U(4π)[0,0]==1", tol=1e-10, absolute=True)
chk(U_2pi_00, -1.0, "spin_half_2pi U(2π)[0,0]==-1", tol=1e-10, absolute=True)

## §9 — Forward and inverse kinematics: numerical methods

In [ ]:
hdr("§9 — Forward and inverse kinematics")

L1_ik = L2_ik = 1.0
x_tgt, y_tgt = 1.0, 1.0

# Analytical IK
r_sq = x_tgt**2 + y_tgt**2
c2_ik = (r_sq - L1_ik**2 - L2_ik**2) / (2*L1_ik*L2_ik)
c2_ik = np.clip(c2_ik, -1, 1)
theta2_ik = np.arccos(c2_ik)  # elbow up
s2_ik = np.sin(theta2_ik)
theta1_ik = np.arctan2(y_tgt, x_tgt) - np.arctan2(L2_ik*s2_ik, L1_ik + L2_ik*c2_ik)
print(f"  Analytical IK: θ1={np.degrees(theta1_ik):.4f}°, θ2={np.degrees(theta2_ik):.4f}°")
print(f"  θ2 should be π/2 = {np.degrees(np.pi/2):.4f}°")

def fk_2dof(th1, th2, l1=1.0, l2=1.0):
    x = l1*np.cos(th1) + l2*np.cos(th1+th2)
    y = l1*np.sin(th1) + l2*np.sin(th1+th2)
    return np.array([x, y])

def jacobian_2dof(th1, th2, l1=1.0, l2=1.0):
    s1_, c1_ = np.sin(th1), np.cos(th1)
    s12, c12 = np.sin(th1+th2), np.cos(th1+th2)
    return np.array([
        [-l1*s1_ - l2*s12, -l2*s12],
        [ l1*c1_ + l2*c12,  l2*c12]
    ])

# Newton-Raphson IK
theta_nr = np.array([0.5, 0.5])
x_target = np.array([x_tgt, y_tgt])
n_iter_nr = 0
for _ in range(100):
    x_cur = fk_2dof(*theta_nr)
    err_vec = x_target - x_cur
    if np.linalg.norm(err_vec) < 1e-9:
        break
    J_nr = jacobian_2dof(*theta_nr)
    J_pinv = np.linalg.pinv(J_nr)
    theta_nr = theta_nr + J_pinv @ err_vec
    n_iter_nr += 1

x_nr = fk_2dof(*theta_nr)
print(f"\n  Newton-Raphson IK: converged in {n_iter_nr} iterations")
print(f"  θ1={np.degrees(theta_nr[0]):.4f}°, θ2={np.degrees(theta_nr[1]):.4f}°")
print(f"  FK verification: x={x_nr[0]:.6f}, y={x_nr[1]:.6f}")

# Workspace plot
fig, ax = plt.subplots(figsize=(6, 6))
angles = np.linspace(-np.pi, np.pi, 500)
# Outer boundary: r = L1+L2 = 2
theta_ws = np.linspace(0, 2*np.pi, 500)
x_outer = 2.0 * np.cos(theta_ws)
y_outer = 2.0 * np.sin(theta_ws)
x_inner = 0.0 * np.cos(theta_ws)  # |L1-L2|=0, degenerate point
ax.plot(x_outer, y_outer, 'b-', linewidth=2, label='r_max = L1+L2 = 2')
ax.plot(0, 0, 'ro', markersize=8, label='r_min = |L1-L2| = 0')

# Scatter sample of workspace
ws_points = []
for t1 in np.linspace(-np.pi, np.pi, 60):
    for t2 in np.linspace(-np.pi, np.pi, 60):
        p = fk_2dof(t1, t2)
        ws_points.append(p)
ws_pts = np.array(ws_points)
ax.scatter(ws_pts[:,0], ws_pts[:,1], s=0.5, alpha=0.15, color='cyan')
ax.set_aspect('equal')
ax.set_title('2-DOF Robot Workspace (L1=L2=1)', fontsize=12)
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
import pathlib
out_dir = pathlib.Path('repl')
out_dir.mkdir(exist_ok=True)
plt.savefig('repl/str_workspace.png', dpi=100)
plt.close()
print("  Saved: repl/str_workspace.png")

workspace_max_r = L1_ik + L2_ik

chk(theta2_ik, np.pi/2, "IK_analytical_theta2 near pi/2", tol=0.001, absolute=True)
chk(np.linalg.norm(x_target - fk_2dof(*theta_nr)), 0.0, "IK_NR_converged ||err||<1e-6", tol=1e-6, absolute=True)
chk(x_nr[0], 1.0, "FK_verification x near 1.0", tol=1e-4, absolute=True)
chk(x_nr[1], 1.0, "FK_verification y near 1.0", tol=1e-4, absolute=True)
chk(workspace_max_r, 2.0, "workspace_max_r == L1+L2 == 2", tol=1e-10, absolute=True)

## §10 — Full loop: symmetry → physics → robotics → quantum control

In [ ]:
hdr("§10 — Full loop: symmetry → physics → robotics → quantum control")

print("""
Symmetry group G
  ├── Discrete: Z_n, D_n → crystal point groups → materials (§1)
  ├── Continuous: SO(2), SO(3) → robot rotations (§5-7)
  │       ↕ same Lie algebra [J_i,J_j]=iε_ijk J_k
  └── Quantum: SU(2) → spin-1/2, Pauli matrices (§8)

Time symmetry T:
  ├── Classical: reversible (§2 harmonic oscillator demo)
  ├── Quantum: T² = -1 for spin-1/2 → Kramers degeneracy
  └── Broken macroscopically: entropy S = k_B ln Ω increases

Noether theorem (§3):
  ├── Time translation → Energy conservation (Hamiltonian)
  ├── Space translation → Momentum conservation
  ├── Rotation SO(3) → Angular momentum conservation
  └── U(1) gauge → Electric charge conservation

Gauge symmetry (§4):
  U(1) → photon → Maxwell → QED
  SU(2) → W±,Z → weak → Higgs → mass
  SU(3) → gluons → QCD → quarks → nuclear binding

Robot arm (§5-9):
  SE(3) = SO(3) ⋉ ℝ³; DH parameters → joint transforms
  Jacobian J → velocity → singularity detection
  IK: Newton-Raphson converges in <20 iterations

D-GS connection:
  GS algorithm = iterative projection in Hilbert space
  Same structure as robot IK iterative solver!
  IK:  ||x_target - f(θ)|| → 0
  GS:  ||I_measured - |u(D)|²|| → 0
""")

# Quantum control: qubit gates via SU(2) rotations
phi_q = symbols('phi', real=True)

# Pauli matrices
sigma_x = Matrix([[0, 1], [1, 0]])
sigma_y = Matrix([[0, -I], [I, 0]])
sigma_z_sym = Matrix([[1, 0], [0, -1]])

# Rotation gates: R_x(φ) = exp(-i φ/2 σ_x) = cos(φ/2)I - i sin(φ/2) σ_x
Rx = cos(phi_q/2)*eye(2) - I*sin(phi_q/2)*sigma_x
Ry = cos(phi_q/2)*eye(2) - I*sin(phi_q/2)*sigma_y
Rz = cos(phi_q/2)*eye(2) - I*sin(phi_q/2)*sigma_z_sym

show(Rx.subs(phi_q, pi), "Rx(π)")

# Hadamard gate
H_gate = (sigma_x + sigma_z_sym) / sqrt(2)
show(H_gate, "Hadamard H = (σ_x + σ_z)/√2")

H_sq = simplify(H_gate * H_gate)
show(H_sq, "H² (should be I)")
H_sq_00 = float(re(H_sq[0, 0]))

# H|0⟩ = |+⟩
ket0 = Matrix([1, 0])
ket_plus = H_gate * ket0
show(ket_plus, "H|0⟩")

# Rx(π) applied to |0⟩
Rx_pi = Rx.subs(phi_q, pi)
Rx_pi_result = Rx_pi * ket0
Rx_pi_imag_1 = float(im(Rx_pi_result[1]))  # should be -1

# Bloch sphere normalization at θ=π/2
bloch_norm = float(Abs(cos(pi/4))**2 + Abs(sin(pi/4))**2)

# GS/IK analogy
gs_ik_analogy = 1  # both converge iteratively

chk(H_sq_00, 1.0, "H_gate_squared [0,0]==1", tol=1e-10, absolute=True)
chk(Rx_pi_imag_1, -1.0, "Rx_pi_applies Im([1])==-1", tol=1e-6, absolute=True)
chk(float(gs_ik_analogy), 1.0, "GS_IK_analogy both_converge==1", tol=1e-10, absolute=True)
chk(bloch_norm, 1.0, "bloch_normalization |cos|²+|sin|²==1", tol=1e-10, absolute=True)

print("\n✓ All sections complete.")